In [0]:
cards_df = spark.table("default.cards_data")

cards_df.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_training.bronze.cards_data")

In [0]:
transactions_df = spark.table("default.transactions_data")

transactions_df.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_training.bronze.transactions_data")

In [0]:
users_df = spark.read.option("header","true").csv(
    "abfss://raw@ticjiahui.dfs.core.windows.net/users_data.csv"
)

users_df.display()

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
users_df.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_training.bronze.users_data")

In [0]:
from pyspark.sql import Row
mcc_df = spark.read \
    .option("multiline", "true") \
    .json("/Volumes/jarvis_training/bronze/train_fraud_labels/mcc_codes.json")

row = mcc_df.first().asDict()

mcc_df = spark.createDataFrame(
    [(k, v) for k, v in row.items()],
    ["mcc_code", "description"]
)

display(mcc_df)


mcc_code,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
%sql
DROP TABLE IF EXISTS jarvis_training.bronze.mcc_codes

In [0]:
mcc_df.write.mode("overwrite").saveAsTable(
    "jarvis_training.bronze.mcc_codes"
)

In [0]:
import json

file_path = "/Volumes/jarvis_training/bronze/train_fraud_labels/train_fraud_labels.json"

json_text = "".join(
    row.value for row in spark.read.text(file_path).collect()
)

data = json.loads(json_text)

fraud_df = spark.createDataFrame(
    [(k, v) for k, v in data["target"].items()],
    ["transaction_id", "fraud_label"]
)

display(fraud_df.limit(10))

transaction_id,fraud_label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
fraud_df.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_training.bronze.train_fraud_labels")